# Deploy LangGraph Agent to Databricks

This notebook provides a simplified, self-contained way to deploy a LangGraph multi-agent system to Databricks.

**Key Features:**
- Hardcoded configurations for simplicity
- Creates and logs the agent to MLflow
- Registers the model in Unity Catalog
- Deploys to Databricks Model Serving
- Includes deployment monitoring and testing

**Prerequisites:**
- Running in a Databricks workspace
- Access to Unity Catalog
- Snowflake secrets configured in Databricks (for SQL workflow)
- Foundation model endpoint available

## 1. Setup Virtual Environment (Local Development Only)

**Note:** This section is for local development. Skip if running in Databricks workspace.

To ensure all dependencies (especially Azure packages for Unity Catalog) are properly installed:
1. Create a virtual environment
2. Activate it
3. Install dependencies from requirements.txt

In [1]:
# Run these commands in your terminal (not needed in Databricks):
# 
# python -m venv venv
# source venv/bin/activate  # On macOS/Linux
# # OR
# venv\Scripts\activate  # On Windows
# 
# pip install -r requirements.txt
#
# Then select this venv as your Python interpreter in VS Code

print("✓ Virtual environment setup instructions above")
print("✓ Make sure your kernel is using the venv Python interpreter")

✓ Virtual environment setup instructions above
✓ Make sure your kernel is using the venv Python interpreter


In [2]:
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import mlflow
from databricks.sdk import WorkspaceClient
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint

# Set MLflow tracking URI to Databricks
mlflow.set_tracking_uri("databricks")

# Enable automatic tracing for LangGraph
mlflow.langchain.autolog()

print("✓ Dependencies imported successfully")
print("✓ LangGraph autologging enabled")

/Users/pragati.gupta/Projects/LanggraphToDatabricks/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Dependencies imported successfully
✓ LangGraph autologging enabled


## 2. Validate Setup

Check that all required files and dependencies are available.

In [3]:
# Validation Cell - Check foundation model access
print("=" * 70)
print("WORKSPACE VALIDATION")
print("=" * 70)

print(f"\nProject root: {Path.cwd()}")

# Check foundation model access
print("\n✓ Checking foundation model access:")
try:
    from databricks.sdk import WorkspaceClient
    ws = WorkspaceClient()
    endpoint = ws.serving_endpoints.get("databricks-meta-llama-3-1-70b-instruct")
    print(f"  ✓ Foundation model accessible: {endpoint.name}")
except Exception as e:
    print(f"  ⚠ Model access check failed: {e}")

print("\n" + "=" * 70)
print("✓ VALIDATION COMPLETE - Ready to proceed!")
print("=" * 70)

WORKSPACE VALIDATION

Project root: /Users/pragati.gupta/Projects/LanggraphToDatabricks

✓ Checking foundation model access:
  ⚠ Model access check failed: Endpoint with name 'databricks-meta-llama-3-1-70b-instruct' does not exist.

✓ VALIDATION COMPLETE - Ready to proceed!


## 2. Configuration (Hardcoded)

Update these values according to your Databricks environment:

In [4]:
# =============================================================================
# CONFIGURATION - Update these values for your environment
# =============================================================================

# Catalog and Schema Configuration
CATALOG_NAME = "databricks_skeleton_rag_dev"  # Your Unity Catalog name - UPDATE THIS
GOLD_SCHEMA = "langgraph_in_databricks"  # Your schema name - UPDATE THIS
RESOURCE_PREFIX = ""  # Optional prefix for resources
USER_NAME = "pragati.gupta"  # Current user (extracted from workspace path)

# Model Configuration
FOUNDATION_MODEL_NAME = "databricks-meta-llama-3-1-70b-instruct"  # Foundation model endpoint
MULTIAGENT_MODEL_NAME = f"{CATALOG_NAME}.{GOLD_SCHEMA}.sql_workflow_agent"  # Registered model name
ENDPOINT_NAME = "sql-workflow-agent-endpoint"  # Serving endpoint name (no underscores)

# Experiment Configuration  
EXPERIMENT_ID = "2869254854362489"  # Your MLflow experiment ID

print("=" * 70)
print("CONFIGURATION LOADED")
print("=" * 70)
print(f"Model: {MULTIAGENT_MODEL_NAME}")
print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Foundation Model: {FOUNDATION_MODEL_NAME}")


CONFIGURATION LOADED
Model: databricks_skeleton_rag_dev.langgraph_in_databricks.sql_workflow_agent
Endpoint: sql-workflow-agent-endpoint
Experiment ID: 2869254854362489
Foundation Model: databricks-meta-llama-3-1-70b-instruct


## 3. Create Agent Script

This cell creates the agent script that will be logged and deployed. The agent should be defined in a separate Python file that MLflow can package.

In [5]:
# Create the agent script file
# This references the sql_workflow_databricks.py file in your workspace

agent_script_content = f'''
import sys
from pathlib import Path

# Add the parent directory to the path to find sql_workflow package
parent_dir = Path(__file__).parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

# Import the SQL workflow orchestrator from your workspace
from sql_workflow.sql_workflow_databricks import SQLWorkflowOrchestrator

# Initialize the SQL Workflow Orchestrator with the foundation model
orchestrator = SQLWorkflowOrchestrator(model="{FOUNDATION_MODEL_NAME}")

# Set as MLflow agent model
import mlflow
mlflow.models.set_model(orchestrator)
'''

# Write the agent script to a temporary file
agent_file_path = Path("sql_workflow_agent.py")
with open(agent_file_path, "w") as f:
    f.write(agent_script_content)

print("=" * 70)
print("✓ Agent script created successfully")
print("=" * 70)
print(f"Script location: {agent_file_path}")
print(f"Foundation model: {FOUNDATION_MODEL_NAME}")
print("=" * 70)

✓ Agent script created successfully
Script location: sql_workflow_agent.py
Foundation model: databricks-meta-llama-3-1-70b-instruct


## 4. Log Model to MLflow

Log the agent to MLflow with all dependencies and resources.

In [6]:
# Create or get MLflow experiment
experiment = mlflow.set_experiment(experiment_id=EXPERIMENT_ID)
print(f"✓ Using experiment: {experiment.name} (ID: {experiment.experiment_id})")

# Prepare resources that the agent needs access to
resources = [
    DatabricksServingEndpoint(endpoint_name=FOUNDATION_MODEL_NAME),
    DatabricksFunction(function_name="system.ai.python_exec"),
]

print(f"✓ Configured {len(resources)} resources")

# Prepare sample request for input example (Agent Framework format)
input_example = {
    "input": [
        {
            "role": "user",
            "content": "Which customer placed the maximum orders in the last month?",
        }
    ]
}

# Create run name with timestamp
job_run_ts = datetime.now(tz=timezone.utc).strftime("%Y%m%d-%H%M%S")
run_name = f"sql-workflow-run-{job_run_ts}"

print(f"✓ Starting MLflow run: {run_name}")

✓ Using experiment: /Users/pragati.gupta@solita.fi/sql-workflow-experiments (ID: 2869254854362489)
✓ Configured 2 resources
✓ Starting MLflow run: sql-workflow-run-20260304-132454


In [7]:
# Log the model to MLflow following Agent Framework best practices
with mlflow.start_run(run_name=run_name, log_system_metrics=True) as run:
    # Get the project root (where this notebook is located)
    project_root = Path.cwd()
    
    print(f"Project root: {project_root}")
    print(f"Code paths:")
    print(f"  - {project_root / 'sql_workflow'}")
    
    # Log model with all necessary code and dependencies
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model=str(agent_file_path),
        code_paths=[
            str(project_root / "sql_workflow"),  # Main workflow package
        ],
        pip_requirements=str(project_root / "requirements.txt"),  # Reference requirements file
        input_example=input_example,
        resources=resources,
    )
    
    print("\n" + "=" * 70)
    print("✓ MODEL LOGGED SUCCESSFULLY")
    print("=" * 70)
    print(f"Run ID: {run.info.run_id}")
    print(f"Model URI: {logged_agent_info.model_uri}")
    print("=" * 70)
    
    # Store for next steps
    RUN_ID = run.info.run_id
    MODEL_URI = logged_agent_info.model_uri

2026/03/04 15:24:55 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/03/04 15:24:55 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Project root: /Users/pragati.gupta/Projects/LanggraphToDatabricks
Code paths:
  - /Users/pragati.gupta/Projects/LanggraphToDatabricks/sql_workflow


2026/03/04 15:24:57 INFO mlflow.pyfunc: Predicting on input example to validate output
2026/03/04 15:24:57 WARNING mlflow.tracing.fluent: Failed to start span predict_stream: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
2026/03/04 15:24:57 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.



🏗️  Building SQL Workflow Graph (Mock Example)
✅ Workflow graph constructed successfully (stops at validation)

📊 Step 1: Schema Helper Agent
✅ Schema retrieved: 3712 characters

🔧 Step 2: SQL Generator with Confidence Scoring
✅ SQL Generated:
   Query: SELECT CUSTOMER_ID, CUSTOMER_NAME, COUNT(*) as order_count FROM ORDERS WHERE ORDER_DATE >= DATE_TRUN...
   Confidence: 75.0%

🚦 Step 3: Confidence Router
🟢 Confidence 75.0% ≥ 70% → Proceeding to Validation

🔍 Step 5: SQL Validator
✅ SQL validation passed


2026/03/04 15:25:08 WARNING mlflow.tracing.fluent: Failed to start span predict_stream: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
2026/03/04 15:25:08 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.



🏗️  Building SQL Workflow Graph (Mock Example)
✅ Workflow graph constructed successfully (stops at validation)

📊 Step 1: Schema Helper Agent
✅ Schema retrieved: 3712 characters

🔧 Step 2: SQL Generator with Confidence Scoring


2026/03/04 15:25:15 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


✅ SQL Generated:
   Query: SELECT CUSTOMER_ID, CUSTOMER_NAME, COUNT(*) as order_count FROM ORDERS WHERE ORDER_DATE >= DATE_TRUN...
   Confidence: 75.0%

🚦 Step 3: Confidence Router
🟢 Confidence 75.0% ≥ 70% → Proceeding to Validation

🔍 Step 5: SQL Validator
✅ SQL validation passed

✓ MODEL LOGGED SUCCESSFULLY
Run ID: b75474ae86f349a391e092ffecde93ef
Model URI: models:/m-8190aa6810384bc18e3be6ed5c2f59b1
🏃 View run sql-workflow-run-20260304-132454 at: https://adb-7405609163114833.13.azuredatabricks.net/ml/experiments/2869254854362489/runs/b75474ae86f349a391e092ffecde93ef
🧪 View experiment at: https://adb-7405609163114833.13.azuredatabricks.net/ml/experiments/2869254854362489


2026/03/04 15:25:18 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/04 15:25:18 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


## 5. Test Model Locally (Optional)

Test the model locally before registering and deploying.

In [8]:
# Optional: Test the model locally
try:
    print("Testing model locally...")
    agent = mlflow.pyfunc.load_model(MODEL_URI)
    sample_response = agent.predict(input_example)
    
    print("\n" + "=" * 70)
    print("✓ MODEL TEST SUCCESSFUL")
    print("=" * 70)
    print(f"Request: {input_example['input'][0]['content']}")
    print(f"\nFull Response:")
    print(sample_response)
    print("=" * 70)
    
except Exception as e:
    print(f"⚠ Local test failed (this may be OK): {e}")
    print("Continuing with registration...")

Testing model locally...



🏗️  Building SQL Workflow Graph (Mock Example)
✅ Workflow graph constructed successfully (stops at validation)

📊 Step 1: Schema Helper Agent
✅ Schema retrieved: 3712 characters

🔧 Step 2: SQL Generator with Confidence Scoring

Provider List: https://docs.litellm.ai/docs/providers

✅ SQL Generated:
   Query: SELECT CUSTOMER_ID, CUSTOMER_NAME, COUNT(*) as order_count FROM ORDERS WHERE ORDER_DATE >= DATE_TRUN...
   Confidence: 75.0%

🚦 Step 3: Confidence Router
🟢 Confidence 75.0% ≥ 70% → Proceeding to Validation

🔍 Step 5: SQL Validator
✅ SQL validation passed

✓ MODEL TEST SUCCESSFUL
Request: Which customer placed the maximum orders in the last month?

Full Response:
{'object': 'response', 'output': [{'type': 'message', 'id': '402e47aa-2f13-4512-9851-8cfc26a5a67b', 'content': [{'text': "**SQL Generated** (Confidence: 75.0%)\n```sql\nSELECT CUSTOMER_ID, CUSTOMER_NAME, COUNT(*) as order_count FROM ORDERS WHERE ORDER_DATE >= DATE_TRUNC('MONTH', DATEADD('MONTH', -1, CURRENT_DATE())) AND OR

## 6. Register Model in Unity Catalog

In [ ]:
# Register the model in Unity Catalog
mlflow.set_registry_uri("databricks-uc")

print(f"Registering model to Unity Catalog: {MULTIAGENT_MODEL_NAME}")

# Register the model using mlflow.register_model (recommended approach)
uc_model_info = mlflow.register_model(
    model_uri=MODEL_URI,
    name=MULTIAGENT_MODEL_NAME
)

MODEL_VERSION = uc_model_info.version

print(f"✓ Model registered as version {MODEL_VERSION}")
print(f"  - Name: {MULTIAGENT_MODEL_NAME}")
print(f"  - Version: {MODEL_VERSION}")

## 7. Deploy to Model Serving Endpoint

Deploy the registered model to a Databricks Model Serving endpoint with environment variables for Snowflake access.

In [ ]:
# Deploy the model to a serving endpoint
print(f"Deploying model version {MODEL_VERSION} to endpoint '{ENDPOINT_NAME}'...")

try:
    # Import agents module inline
    from databricks import agents
    
    agents.deploy(
        model_name=MULTIAGENT_MODEL_NAME,
        model_version=int(MODEL_VERSION),
        endpoint_name=ENDPOINT_NAME,
    )
    
    print("✓ Deployment request submitted successfully")
    print(f"  - Endpoint: {ENDPOINT_NAME}")
    print(f"  - Model: {MULTIAGENT_MODEL_NAME} v{MODEL_VERSION}")
    
except ImportError:
    print("⚠ databricks.agents module not available")
    print("  Deploy manually using Databricks UI or workspace client")
    print(f"  Model: {MULTIAGENT_MODEL_NAME} v{MODEL_VERSION}")
except Exception as e:
    print(f"✗ Deployment failed: {e}")
    raise

## 8. Wait for Endpoint to be Ready

Monitor the deployment progress until the endpoint is ready to serve requests.

In [ ]:
def wait_for_endpoint_ready(endpoint_name: str, timeout_seconds: int = 1800, poll_interval: int = 30):
    """
    Poll serving endpoint until deployment is complete.
    
    Args:
        endpoint_name: Name of the serving endpoint
        timeout_seconds: Maximum time to wait (default: 1800 = 30 min)
        poll_interval: Time between status checks (default: 30s)
    """
    ws = WorkspaceClient()
    start_time = time.time()
    
    print(f"⏳ Waiting for endpoint '{endpoint_name}' to be ready...")
    print(f"   (timeout: {timeout_seconds}s, polling every {poll_interval}s)")
    
    while time.time() - start_time < timeout_seconds:
        try:
            endpoint = ws.serving_endpoints.get(endpoint_name)
            
            config_update = endpoint.state.config_update.name
            ready_state = endpoint.state.ready.name
            
            elapsed = int(time.time() - start_time)
            print(f"   [{elapsed}s] Status: config_update={config_update}, ready={ready_state}")
            
            # Check if update is complete
            if config_update == "NOT_UPDATING":
                if ready_state == "READY":
                    print(f"✓ Endpoint '{endpoint_name}' is READY and serving!")
                    return True
                else:
                    print(f"✗ Deployment failed with state: {ready_state}")
                    raise Exception(f"Deployment failed with state: {ready_state}")
            
            # Still updating, wait before next poll
            time.sleep(poll_interval)
            
        except Exception as e:
            if "does not exist" in str(e):
                print(f"✗ Endpoint '{endpoint_name}' not found")
                raise
            raise
    
    # Timeout reached
    elapsed = int(time.time() - start_time)
    raise TimeoutError(f"Endpoint deployment timeout after {elapsed}s")

# Wait for the endpoint to be ready
wait_for_endpoint_ready(ENDPOINT_NAME)

## 9. Test the Deployed Endpoint

Send a test query to verify the endpoint is working correctly.

In [ ]:
# Test the deployed endpoint
print(f"Testing endpoint: {ENDPOINT_NAME}")

# Create a test request (Agent Framework format)
test_request = {
    "input": [
        {
            "role": "user",
            "content": "What are the top 5 customers by order volume?"
        }
    ]
}

try:
    # Query the endpoint using Databricks SDK
    # For Agent Framework models, use dataframe_records parameter
    ws = WorkspaceClient()
    response = ws.serving_endpoints.query(
        name=ENDPOINT_NAME,
        dataframe_records=[test_request]
    )
    
    print("✓ Endpoint test successful!")
    print(f"  Request: {test_request['input'][0]['content']}")
    print(f"  Response: {response}")
    
except Exception as e:
    print(f"✗ Endpoint test failed: {e}")
    print("  The endpoint may still be initializing. Try again in a few minutes.")

## 10. Delete Endpoint (Optional)

If you want to clean up and delete the endpoint after testing, run the cell below. **Warning:** This will permanently delete the endpoint.

In [ ]:
# Delete the serving endpoint
# Uncomment the code below to delete the endpoint

print(f"⚠️  Deleting endpoint: {ENDPOINT_NAME}")
print("This action cannot be undone!")

try:
    ws = WorkspaceClient()
    ws.serving_endpoints.delete(name=ENDPOINT_NAME)
    
    print(f"✓ Endpoint '{ENDPOINT_NAME}' has been deleted successfully")
    print("  The endpoint and its resources have been removed")
    
except Exception as e:
    print(f"✗ Failed to delete endpoint: {e}")
    print("  The endpoint may not exist or you may not have permission to delete it")

print("⚠️  Endpoint deletion is commented out by default")
print("Uncomment the code above if you want to delete the endpoint")

## 11. Summary and Next Steps

**Deployment Complete!** 🎉

Your LangGraph agent has been successfully deployed to Databricks Model Serving.

**What was deployed:**
- Model: `{MULTIAGENT_MODEL_NAME}` version `{MODEL_VERSION}`
- Endpoint: `{ENDPOINT_NAME}`
- Foundation Model: `{FOUNDATION_MODEL_NAME}`

**Next steps:**
1. **Monitor the endpoint** in the Databricks Serving UI
2. **Set up Review App** for human feedback (optional)
3. **Configure auto-scaling** based on traffic patterns
4. **Set up alerts** for endpoint health monitoring
5. **Test with production queries** to validate performance

**To query the endpoint from code:**
```python
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
response = ws.serving_endpoints.query(
    name="{ENDPOINT_NAME}",
    dataframe_records=[{"input": [{"role": "user", "content": "Your question here"}]}]
)
```

**Useful links:**
- Endpoint URL: `https://<workspace-url>/serving-endpoints/{ENDPOINT_NAME}/invocations`
- MLflow Experiment: `{EXPERIMENT_PATH}`
- Model Registry: Unity Catalog → `{CATALOG_NAME}` → `{GOLD_SCHEMA}` → Models

**Cleanup:**
- To delete the endpoint, run the previous cell (section 10)